# Single-author papers and repeated author lists

Section 4.2.2 and Figure 3. Compute the reported authorship proportions, repeated-list pre/post interaction, repeated-author mean paper share among authors with at least five papers, and the requested citation-count contrasts. No exploratory sensitivity analyses are included. All computations are deterministic.

The relative default `data/` directory contains `{pred,norm}/dataset/csv/papers.csv` and `papers_collect_after_6months.csv`. Required columns are `id`, `authors`, `publication_date` in the paper table and `old_id`, `total_citations` in the follow-up table. Results default to `outputs/03_authorship_patterns/`. Data and generated results are not included with the code.

In [ ]:
from pathlib import Path
import os

REPO_ROOT = Path('..') if Path.cwd().name == 'notebooks' else Path('.')
DATA_DIR = REPO_ROOT / 'data'
OUTPUT_DIR = REPO_ROOT / 'outputs' / '03_authorship_patterns'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
import math
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import patsy
import statsmodels.api as sm
from scipy.stats import norm
from statsmodels.stats.proportion import proportion_confint, proportions_ztest

PRED_COLOR = '#a3245d'
NORM_COLOR = '#3e99bb'
LINE_COLOR = '#555555'
PERIOD_ORDER = ['Overall', 'Before 2023', '2023–2024']

## Eligibility and authorship definitions

Require nonmissing authors and publication date. Extract the first four-digit year and retain 2018–2024. The pre-period is 2018–2022 and the post-period is 2023–2024.

For single-author classification, normalize semicolons, the word separator ` and `, and the full-width comma to commas, strip components, and count remaining components. For repeated lists, preserve the original ordered author string: it must contain an ordinary comma and occur on at least two papers within the same dataset and calendar year. Count every paper carrying that list. Thus repetition across different calendar years alone does not qualify. The two parsing rules deliberately preserve the source definitions; author spellings are not disambiguated.

In [ ]:
def normalize_delimiters(value):
    value = str(value).replace(';', ',').replace(' and ', ',').replace('，', ',')
    return ','.join(part.strip() for part in value.split(',') if part.strip())

def load_papers(group, dataset):
    raw = pd.read_csv(DATA_DIR/group/'dataset/csv/papers.csv', usecols=['id','authors','publication_date'])
    assert raw.id.notna().all() and raw.id.is_unique
    d = raw.dropna(subset=['authors','publication_date']).copy()
    d['year'] = pd.to_numeric(d.publication_date.astype(str).str.extract(r'(\d{4})', expand=False), errors='coerce')
    d = d.dropna(subset=['year']).copy()
    d['year'] = d.year.astype(int)
    d = d[d.year.between(2018,2024)].copy()
    d['dataset'] = dataset
    d['single_author'] = d.authors.map(normalize_delimiters).str.split(',').map(len).eq(1).astype(int)
    frequency = d.groupby(['year','authors']).authors.transform('size')
    d['repeated_authorlist'] = (d.authors.str.contains(',', regex=False) & frequency.ge(2)).astype(int)
    d['period'] = np.where(d.year.le(2022), 'Before 2023', '2023–2024')
    return d

frames = {dataset:load_papers(group,dataset) for dataset,group in [('Predatory','pred'),('Normal','norm')]}
authorship = pd.concat(frames.values(),ignore_index=True)

## Overall proportions and Figure 3

Report observed counts and denominators for each period, with Wilson 95% intervals for the percentage bars. The overall two-sided pooled proportion z-test and signed Cohen's h use Predatory minus Normal. Pre/post gaps are descriptive differences in percentage points.

In [ ]:
period_rows = []
for outcome in ['single_author','repeated_authorlist']:
    for period in PERIOD_ORDER:
        row = dict(outcome=outcome, period=period)
        for dataset,prefix in [('Predatory','pred'),('Normal','norm')]:
            d = frames[dataset]
            if period != 'Overall':
                d = d[d.period.eq(period)]
            count, denominator = int(d[outcome].sum()),len(d)
            low,high = proportion_confint(count,denominator,method='wilson')
            row.update({prefix+'_count':count,prefix+'_denominator':denominator,
                prefix+'_proportion':count/denominator,prefix+'_ci_low':low,prefix+'_ci_high':high})
        row['difference_pp'] = 100*(row['pred_proportion']-row['norm_proportion'])
        period_rows.append(row)
period_results = pd.DataFrame(period_rows)
overall_tests = []
for row in period_results[period_results.period.eq('Overall')].to_dict('records'):
    z,p = proportions_ztest([row['pred_count'],row['norm_count']], [row['pred_denominator'],row['norm_denominator']], alternative='two-sided')
    overall_tests.append(dict(outcome=row['outcome'],z=z,p_value=p,
        cohen_h=2*(np.arcsin(np.sqrt(row['pred_proportion']))-np.arcsin(np.sqrt(row['norm_proportion'])))))
period_results.to_csv(OUTPUT_DIR/'authorship_counts_and_rates.csv',index=False)
pd.DataFrame(overall_tests).to_csv(OUTPUT_DIR/'overall_tests.csv',index=False)
print(period_results[['outcome','period','pred_count','pred_denominator','pred_proportion','norm_count','norm_denominator','norm_proportion','difference_pp']].to_string(index=False))
print(pd.DataFrame(overall_tests).to_string(index=False))
single_results = period_results[period_results.outcome.eq('single_author')].reset_index(drop=True)
repeated_results = period_results[period_results.outcome.eq('repeated_authorlist')].reset_index(drop=True)

In [ ]:
def plot_results(single_results, repeated_results, output_dir):
    plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 11, 'axes.labelsize': 12, 'xtick.labelsize': 10, 'ytick.labelsize': 11, 'legend.fontsize': 12})
    fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.15), sharey=True)
    panels = [(axes[0], single_results, 'A'), (axes[1], repeated_results, 'B')]
    all_upper = np.concatenate([single_results[['pred_ci_high', 'norm_ci_high']].to_numpy().ravel(), repeated_results[['pred_ci_high', 'norm_ci_high']].to_numpy().ravel()])
    ymax = math.ceil((100 * np.nanmax(all_upper) + 5) / 5) * 5
    x = np.arange(len(PERIOD_ORDER))
    width = 0.3
    gap_offset = 0.19
    for ax, results, panel in panels:
        for offset, prefix, label, color, hatch in [(-gap_offset, 'pred', 'Predatory', PRED_COLOR, None), (gap_offset, 'norm', 'Normal', NORM_COLOR, '\\\\')]:
            values = 100 * results[f'{prefix}_proportion'].to_numpy()
            low = 100 * results[f'{prefix}_ci_low'].to_numpy()
            high = 100 * results[f'{prefix}_ci_high'].to_numpy()
            bars = ax.bar(x + offset, values, width, label=label, color=color, edgecolor='black', linewidth=0.6, hatch=hatch, zorder=2)
            ax.errorbar(x + offset, values, yerr=np.vstack([values - low, high - values]), fmt='none', ecolor=LINE_COLOR, elinewidth=1.3, capsize=3.5, capthick=1.3, zorder=3)
            for bar, value, upper in zip(bars, values, high):
                ax.text(bar.get_x() + bar.get_width() / 2, upper + ymax * 0.04, f'{value:.1f}%', ha='center', va='bottom', fontsize=9)
        ax.set_xticks(x)
        ax.set_xticklabels(['Overall', 'Before\n2023', '2023–\n2024'])
        ax.set_ylim(0, ymax)
        ax.text(-0.12, 1.14, panel, transform=ax.transAxes, fontweight='bold', fontsize=12, va='top')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.grid(False)
    axes[0].set_ylabel('Percentage of papers')
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper right', bbox_to_anchor=(0.98, 0.9), frameon=False, ncol=2)
    fig.tight_layout(rect=(0, 0, 1, 0.92), w_pad=0.6)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    pdf_path = output_dir / 'fig3_authorship_patterns.pdf'
    png_path = output_dir / 'fig3_authorship_patterns.png'
    fig.savefig(pdf_path, bbox_inches='tight', metadata={'Creator':'Analysis notebook','CreationDate':None,'ModDate':None})
    fig.savefig(png_path, dpi=600, bbox_inches='tight')
    plt.show()
    return (pdf_path.resolve(), png_path.resolve())
figure_paths = plot_results(single_results,repeated_results,OUTPUT_DIR)
plt.close('all')

## Repeated-list logistic interaction

Fit a binomial-logit model with indicators for Predatory, post-period, and their product. Cluster standard errors by the exact raw author-list string across years and datasets, preserving the source clustered specification. Report the exponentiated interaction coefficient: the Predatory post/pre odds ratio divided by the Normal post/pre odds ratio. It is not the overall between-set odds ratio.

In [ ]:
model_data = authorship.copy()
model_data['dataset_predatory'] = model_data.dataset.eq('Predatory').astype(int)
model_data['post_period'] = model_data.year.ge(2023).astype(int)
model_data['interaction'] = model_data.dataset_predatory*model_data.post_period
design = sm.add_constant(model_data[['dataset_predatory','post_period','interaction']],has_constant='add')
clusters = pd.factorize(model_data.authors,sort=True)[0]
logistic = sm.GLM(model_data.repeated_authorlist,design,family=sm.families.Binomial()).fit(cov_type='cluster',cov_kwds={'groups':clusters})
assert logistic.converged
low,high = logistic.conf_int().loc['interaction']
interaction_result = pd.DataFrame([dict(parameter='Predatory x post-period',n=int(logistic.nobs),clusters=len(np.unique(clusters)),odds_ratio=np.exp(logistic.params['interaction']),ci_low=np.exp(low),ci_high=np.exp(high),p_value=logistic.pvalues['interaction'])])
interaction_result.to_csv(OUTPUT_DIR/'repeated_list_interaction.csv',index=False)
print(interaction_result.to_string(index=False))

## Repeated-paper share among authors with at least five papers

Parse author strings using the same delimiter normalization, treating each distinct stripped spelling as a separate author and counting each paper once per spelling. Among authors observed on at least one repeated-list paper and at least five total eligible papers, average each author's repeated-paper/total-paper ratio with equal author weight. This is a mean share of papers, not the percentage of all authors who have five papers. No per-author rows or author strings are exported.

In [ ]:
author_rows = []
for dataset,d in frames.items():
    total,repeated = defaultdict(set),defaultdict(set)
    for row in d.itertuples(index=False):
        for author in set(normalize_delimiters(row.authors).split(',')):
            if not author:
                continue
            total[author].add(row.id)
            if row.repeated_authorlist:
                repeated[author].add(row.id)
    ratios = [len(ids)/len(total[author]) for author,ids in repeated.items() if len(total[author])>=5]
    author_rows.append(dict(dataset=dataset,eligible_authors=len(ratios),mean_repeated_paper_percent=100*np.mean(ratios)))
author_summary = pd.DataFrame(author_rows)
author_summary.to_csv(OUTPUT_DIR/'author_min5_summary.csv',index=False)
print(author_summary.to_string(index=False))

## Requested negative-binomial citation contrasts

These are citation-count contrasts associated with repeated author lists, not models of author-list frequency. Match eligible papers to the follow-up snapshot, excluding missing or invalid nonnegative integer counts without imputation. Fit NB2 with authorship category, dataset, their interaction, and categorical publication year. Cluster covariance by dataset/year/exact-list for repeated papers and by dataset/paper for other papers.

Within each dataset, report repeated-list versus nonrepeated multi-author citation IRRs, using the source normal-approximation interval. Single-author papers are a separate category. No citation-growth, profile-credit, alternative-reference or additional sensitivity analysis is performed.

In [ ]:
citation_frames = []
for dataset,group in [('Predatory','pred'),('Normal','norm')]:
    d = frames[dataset].copy()
    snapshot = pd.read_csv(DATA_DIR/group/'dataset/csv/papers_collect_after_6months.csv',usecols=['old_id','total_citations']).rename(columns={'old_id':'id','total_citations':'citation_count'})
    assert snapshot.id.notna().all() and snapshot.id.is_unique
    d = d.merge(snapshot,on='id',how='left',validate='one_to_one')
    count = pd.to_numeric(d.citation_count,errors='coerce')
    d = d[count.notna() & np.isfinite(count) & count.ge(0) & count.eq(np.floor(count))].copy()
    d['citation_count'] = pd.to_numeric(d.citation_count)
    d['authorship_type'] = np.select([d.single_author.eq(1),d.repeated_authorlist.eq(1)],['single_author','repeated_authorlist'],default='nonrepeated_multi_author')
    d['publication_year'] = d.year
    d['cluster_id'] = np.where(d.repeated_authorlist.eq(1),dataset+'|'+d.year.astype(str)+'|'+d.authors,dataset+'|unique|'+d.id.astype(str))
    citation_frames.append(d)
citation_data = pd.concat(citation_frames,ignore_index=True)
formula = "citation_count ~ C(authorship_type, Treatment(reference='nonrepeated_multi_author')) * C(dataset, Treatment(reference='Normal')) + C(publication_year)"
response,design = patsy.dmatrices(formula,citation_data,return_type='dataframe',NA_action='raise')
negative_binomial = sm.NegativeBinomial(response.iloc[:,0],design).fit(disp=0,maxiter=200,cov_type='cluster',cov_kwds={'groups':citation_data.cluster_id})
assert negative_binomial.mle_retvals['converged'] and np.isfinite(negative_binomial.params).all()
nb_rows = []
for dataset in ['Predatory','Normal']:
    reference = pd.DataFrame(dict(authorship_type=['repeated_authorlist','nonrepeated_multi_author'],dataset=[dataset,dataset],publication_year=[2022,2022]))
    encoded = np.asarray(patsy.build_design_matrices([design.design_info],reference)[0])
    contrast = encoded[0]-encoded[1]
    estimate = float(contrast@negative_binomial.params.reindex(design.columns))
    se = float(np.sqrt(contrast@negative_binomial.cov_params().loc[design.columns,design.columns]@contrast))
    nb_rows.append(dict(dataset=dataset,n=int(negative_binomial.nobs),irr=np.exp(estimate),ci_low=np.exp(estimate-1.96*se),ci_high=np.exp(estimate+1.96*se),p_value=2*norm.sf(abs(estimate/se))))
nb_results = pd.DataFrame(nb_rows)
nb_results.to_csv(OUTPUT_DIR/'citation_irrs.csv',index=False)
print(nb_results.to_string(index=False))